In [ ]:
from pyspark.sql import SparkSession
spark=SparkSession.builder \
    .appName("Working with joins") \
    .getOrCreate()

In [ ]:
customers = [

(1, "Rahul", "Hyderabad"),
(2, "Priya", "Bangalore"),
(3, "Amit", "Mumbai"),
(4, "Sneha", "Chennai"),
(5, "Farhan", "Delhi")

]
customer_columns = [
    "customer_id",
    "customer_name",
    "city"
]
customers_df = spark.createDataFrame(
    customers,
    customer_columns
)
customers_df.show()

+-----------+-------------+---------+
|customer_id|customer_name|     city|
+-----------+-------------+---------+
|          1|        Rahul|Hyderabad|
|          2|        Priya|Bangalore|
|          3|         Amit|   Mumbai|
|          4|        Sneha|  Chennai|
|          5|       Farhan|    Delhi|
+-----------+-------------+---------+



In [ ]:
orders = [

(101, 1, "Laptop", 65000),
(102, 2, "Mobile", 25000),
(103, 1, "TV", 45000),
(104, 3, "Chair", 5000),
(105, 7, "Watch", 8000)
]
order_columns = [
    "order_id",
    "customer_id",
    "product",
    "amount"
]
orders_df = spark.createDataFrame(
    orders,
    order_columns
)
orders_df.show()

+--------+-----------+-------+------+
|order_id|customer_id|product|amount|
+--------+-----------+-------+------+
|     101|          1| Laptop| 65000|
|     102|          2| Mobile| 25000|
|     103|          1|     TV| 45000|
|     104|          3|  Chair|  5000|
|     105|          7|  Watch|  8000|
+--------+-----------+-------+------+



In [ ]:
customers_df.join(
    orders_df,
    "customer_id",
    "inner"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [ ]:
customers_df.join(
    orders_df,
    "customer_id",
    "left"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          5|       Farhan|    Delhi|    NULL|   NULL|  NULL|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
|          4|        Sneha|  Chennai|    NULL|   NULL|  NULL|
+-----------+-------------+---------+--------+-------+------+



In [ ]:
customers_df.join(
    orders_df,
    "customer_id",
    "right"
).show()

+-----------+-------------+---------+--------+-------+------+
|customer_id|customer_name|     city|order_id|product|amount|
+-----------+-------------+---------+--------+-------+------+
|          1|        Rahul|Hyderabad|     101| Laptop| 65000|
|          2|        Priya|Bangalore|     102| Mobile| 25000|
|          7|         NULL|     NULL|     105|  Watch|  8000|
|          1|        Rahul|Hyderabad|     103|     TV| 45000|
|          3|         Amit|   Mumbai|     104|  Chair|  5000|
+-----------+-------------+---------+--------+-------+------+



In [ ]:
from pyspark.sql.functions import sum
customers_df.alias("c").join(
    orders_df.alias("o"),
    "customer_id"
).groupBy(
    "customer_id",
    "customer_name"
).agg(
    sum("o.amount").alias("total_amount")
).show()

+-----------+-------------+------------+
|customer_id|customer_name|total_amount|
+-----------+-------------+------------+
|          1|        Rahul|      110000|
|          2|        Priya|       25000|
|          3|         Amit|        5000|
+-----------+-------------+------------+



In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [ ]:
window_spec=Window.orderBy(
    col("amount").desc()
)
orders_df.withColumn(
    "row_number",
    row_number().over(window_spec)
).show()

+--------+-----------+-------+------+----------+
|order_id|customer_id|product|amount|row_number|
+--------+-----------+-------+------+----------+
|     101|          1| Laptop| 65000|         1|
|     103|          1|     TV| 45000|         2|
|     102|          2| Mobile| 25000|         3|
|     105|          7|  Watch|  8000|         4|
|     104|          3|  Chair|  5000|         5|
+--------+-----------+-------+------+----------+



In [ ]:
window_spec=Window.orderBy(
    col("amount").desc()
)
orders_df.withColumn(
    "rank",
    rank().over(window_spec)
).show()

+--------+-----------+-------+------+----+
|order_id|customer_id|product|amount|rank|
+--------+-----------+-------+------+----+
|     101|          1| Laptop| 65000|   1|
|     103|          1|     TV| 45000|   2|
|     102|          2| Mobile| 25000|   3|
|     105|          7|  Watch|  8000|   4|
|     104|          3|  Chair|  5000|   5|
+--------+-----------+-------+------+----+



In [ ]:
window_spec=Window.orderBy(
    col("amount").desc()
)
orders_df.withColumn(
    "dense_rank",
    dense_rank().over(window_spec)
).show()

+--------+-----------+-------+------+----------+
|order_id|customer_id|product|amount|dense_rank|
+--------+-----------+-------+------+----------+
|     101|          1| Laptop| 65000|         1|
|     103|          1|     TV| 45000|         2|
|     102|          2| Mobile| 25000|         3|
|     105|          7|  Watch|  8000|         4|
|     104|          3|  Chair|  5000|         5|
+--------+-----------+-------+------+----------+



In [ ]:
window_spec=Window.partitionBy(
    "customer_id"
).orderBy(
    col("amount").desc()
)
orders_df.withColumn(
    "customer_rank",
    rank().over(window_spec)
).show()

+--------+-----------+-------+------+-------------+
|order_id|customer_id|product|amount|customer_rank|
+--------+-----------+-------+------+-------------+
|     101|          1| Laptop| 65000|            1|
|     103|          1|     TV| 45000|            2|
|     102|          2| Mobile| 25000|            1|
|     104|          3|  Chair|  5000|            1|
|     105|          7|  Watch|  8000|            1|
+--------+-----------+-------+------+-------------+

